# Vectorized QMC in Julia

Demonstrates QMC integration with vectorized (batched) function evaluation, comparing IID, digital net, and lattice sequences.

In [ ]:
using QMC
import QMC: Uniform
using Statistics

## LD Sequences

Compare IID, Digital Net, and Lattice point sets.

In [ ]:
n = 2^6
for (dd, name) in [
    (IIDStdUniform(2; seed=7),  "IID"),
    (DigitalNetB2(2; seed=7),   "Digital Net"),
    (Lattice(2; seed=7),        "Lattice"),
]
    pts = gen_samples(dd, n)
    println("$name: $(size(pts, 1)) points, " *
            "mean=$(round.(mean(pts, dims=1), digits=3))")
end

## Vectorized Integration

The Cantilever Beam function maps 3D input to 2D output (displacement and stress). QMC.jl naturally handles vectorized evaluation.

In [ ]:
# Cantilever beam: (E, X, Y) → (displacement D, stress S)
function cantilever_beam(x)
    l, w, t = 100.0, 4.0, 2.0
    E, X, Y = x[1], x[2], x[3]
    D = 4l^3 / (E * w * t) * sqrt(X^2 / w^4 + Y^2 / t^4)
    S = 6l / (w * t^2) * sqrt(X^2 + Y^2)
    return [D, S]
end

# Evaluate over QMC points
dd = DigitalNetB2(3; seed=7)
x = gen_samples(dd, 1024)

results = hcat([cantilever_beam(x[i, :]) for i in 1:size(x, 1)]...)
println("Displacement: mean=$(round(mean(results[1,:]), digits=4)), " *
        "std=$(round(std(results[1,:]), digits=4))")
println("Stress:       mean=$(round(mean(results[2,:]), digits=4)), " *
        "std=$(round(std(results[2,:]), digits=4))")

## CubMCCLTVec for Vector Outputs

Use `CubMCCLTVec` to simultaneously integrate all output components.

In [ ]:
# Wrap as CustomFun for integration
dd2 = IIDStdUniform(3; seed=7)
tm2 = Uniform(dd2)
cf = CustomFun(tm2, x -> cantilever_beam(x))
sc = CubMCCLTVec(cf; abs_tol=0.1)
result = integrate(sc)
println("Solution: $(result.solution)")
println("Converged: $(result.data[:converged])")